In [1]:
from google.colab import userdata
import os

aws_dir = os.path.expanduser("~/.aws")
os.makedirs(aws_dir, exist_ok=True)

key_id = userdata.get('AWS_ACCESS_KEY_ID')
secret_key = userdata.get('AWS_SECRET_ACCESS_KEY')

with open(f"{aws_dir}/credentials", "w") as f:
    f.write(f"[default]\naws_access_key_id = {key_id}\naws_secret_access_key = {secret_key}\n")

with open(f"{aws_dir}/config", "w") as f:
    f.write("[default]\nregion = us-east-1\n")

print("Successfully configured ~/.aws/credentials and ~/.aws/config")

Successfully configured ~/.aws/credentials and ~/.aws/config


In [2]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [359 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:7 http://archive.ubuntu.com/ubuntu noble InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,878 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,030 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/restrict

In [3]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Sep 29, 2026 10:10:32 AM java.util.prefs.FileSystemPreferences$1 run
INFO: Created user preferences directory.
->   ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  Python interpreter initialized safely.


In [4]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path ; from IPython.display import display, HTML
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists(): os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_13/class1/class1'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


->  Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
>>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Cloning into '/content/BITS_programming'...
Updating files:  11% (165/1417)
Updating files:  12% (171/1417)
Updating files:  13% (185/1417)
Updating files:  14% (199/1417)
Updating files:  15% (213/1417)
Updating files:  16% (227/1417)
Updating files:  17% (241/1417)
Updating files:  18% (256/1417)
Updating files:  19% (270/1417)
Updating files:  20% (284/1417)
Updating files:  21% (298/1417)
Updating files:  22% (312/1417)
Updating files:  23% (326/1417)
Updating files:  24% (341/1417)
Updating files:  25% (355/1417)
Updating files:  26% (369/1417)
Updating files:  27% (383/1417)
Updating files:  28% (397/1417)
Updating files:  29% (411/1417)
Updating files:  30% (426/1417)
Updating files:  31% (440/1417)
Updating files:  32% (454/1417)
Updating files:  33% (468/1417)
Updating files

# Lab 1: CNN Robustness Report - SOLUTION NOTEBOOK

## Objective
Build a comprehensive CNN robustness report by:
1. Training a baseline CNN model
2. Detecting overfitting from training curves
3. Applying 2 regularization techniques and comparing
4. Running adversarial attacks (FGSM)
5. Generating a final robustness report

## Complete Implementation
This notebook contains the full solution with detailed explanations.

## Section 1: Setup & Data Loading

In [5]:
%%jshell

runPy("""
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from IPython.display import HTML, display
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18
from tqdm import tqdm

# Install required packages cleanly
os.system('python -m pip install torchvision torch tqdm seaborn matplotlib')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✅ Libraries imported")
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Requirement already satisfied: torchvision in /usr/local/lib/python3.13/dist-packages (0.26.0+cu128)
✅ Libraries imported


In [6]:
%%jshell

runPy("""
# Device and paths
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DATASET_PATH = str(REPO_ROOT / "module_4/dataset/dataset")
TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VAL_PATH = os.path.join(DATASET_PATH, "val")
TEST_PATH = os.path.join(DATASET_PATH, "test")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Device: cuda


In [7]:
%%jshell

runPy("""
# Data transformations and loading
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(TRAIN_PATH, transform=transform)
val_dataset = ImageFolder(VAL_PATH, transform=transform)
test_dataset = ImageFolder(TEST_PATH, transform=transform)

class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Classes: {class_names}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Classes: ['animal', 'name_board', 'pedestrian', 'pothole', 'road_sign', 'speed_breaker', 'vehicle']
Train: 796 | Val: 20 | Test: 117


In [8]:
%%jshell

runPy("""
# Create data loaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✅ Data loaders created with batch size {BATCH_SIZE}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ Data loaders created with batch size 64


## Section 2: Define Models with Regularization

In [9]:
%%jshell

runPy("""
# Baseline model (no regularization)
class BaselineModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.resnet = resnet18(pretrained=False)  # Train from scratch to show overfitting
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

print("✅ BaselineModel defined")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ BaselineModel defined


In [10]:
%%jshell

runPy("""
# L2 Regularized Model (identical architecture, but trained with weight decay)
class L2RegularizedModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.resnet = resnet18(pretrained=False)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

print("✅ L2RegularizedModel defined")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ L2RegularizedModel defined


In [11]:
%%jshell

runPy("""
# Dropout Model (adds dropout layer before classification)
class DropoutModel(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.resnet = resnet18(pretrained=False)
        # Replace final layer with Dropout + Linear
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout_rate),  # Randomly drops 50% of neurons
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.resnet(x)

print("✅ DropoutModel defined")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ DropoutModel defined


## Section 3: Training & Evaluation Functions

In [12]:
%%jshell

runPy("""
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Track metrics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("✅ train_epoch() defined")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ train_epoch() defined


In [13]:
%%jshell

runPy("""
def evaluate(model, loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("✅ evaluate() defined")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ evaluate() defined


## Section 4: Train All Three Models

In [14]:
%%jshell

runPy("""
# Train Baseline Model (No Regularization)
print("\\n" + "="*60)
print("Training BASELINE Model (No Regularization)")
print("="*60)

baseline_model = BaselineModel(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_baseline = optim.Adam(baseline_model.parameters(), lr=0.001, weight_decay=0)  # No weight decay

baseline_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
NUM_EPOCHS = 30

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(baseline_model, train_loader, criterion, optimizer_baseline, device)
    val_loss, val_acc = evaluate(baseline_model, val_loader, criterion, device)

    baseline_history['train_loss'].append(train_loss)
    baseline_history['train_acc'].append(train_acc)
    baseline_history['val_loss'].append(val_loss)
    baseline_history['val_acc'].append(val_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("✅ Baseline model trained")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
Training BASELINE Model (No Regularization)
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)

Training: 100%|██████████| 13/13 [00:05<00:00,  3.67it/s]
                                                         

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  5.83it/s]
                                                         

Training: 100%|██████████| 13/13 [00:03<00:00,  4.94it/s]
                                                         

Evaluating: 100%|█████████

In [15]:
%%jshell

runPy("""
# Train L2 Regularized Model
print("\\n" + "="*60)
print("Training L2 REGULARIZED Model (Weight Decay = 0.001)")
print("="*60)

l2_model = L2RegularizedModel(num_classes).to(device)
optimizer_l2 = optim.Adam(l2_model.parameters(), lr=0.001, weight_decay=0.001)  # L2 regularization

l2_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(l2_model, train_loader, criterion, optimizer_l2, device)
    val_loss, val_acc = evaluate(l2_model, val_loader, criterion, device)

    l2_history['train_loss'].append(train_loss)
    l2_history['train_acc'].append(train_acc)
    l2_history['val_loss'].append(val_loss)
    l2_history['val_acc'].append(val_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("✅ L2 Regularized model trained")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
Training L2 REGULARIZED Model (Weight Decay = 0.001)

Training: 100%|██████████| 13/13 [00:03<00:00,  3.67it/s]
                                                         

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  6.63it/s]
                                                         

Training: 100%|██████████| 13/13 [00:03<00:00,  4.49it/s]
                                                         

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  9.95it/s]
                                                         

Training: 100%|██████████| 13/13 [00:03<00:00,  4.24it/s]
                                                         

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  9.72it/s]
                                                         

Training: 100%|██████████| 13/13 [00:03<00:00,  4.38it/s]
                                                         

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  9.99it/s]


In [16]:
%%jshell

runPy("""
# Train Dropout Model
print("\\n" + "="*60)
print("Training DROPOUT Model (Dropout Rate = 0.5)")
print("="*60)

dropout_model = DropoutModel(num_classes, dropout_rate=0.5).to(device)
optimizer_dropout = optim.Adam(dropout_model.parameters(), lr=0.001, weight_decay=0)

dropout_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(dropout_model, train_loader, criterion, optimizer_dropout, device)
    val_loss, val_acc = evaluate(dropout_model, val_loader, criterion, device)

    dropout_history['train_loss'].append(train_loss)
    dropout_history['train_acc'].append(train_acc)
    dropout_history['val_loss'].append(val_loss)
    dropout_history['val_acc'].append(val_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("✅ Dropout model trained")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
Training DROPOUT Model (Dropout Rate = 0.5)

Training: 100%|██████████| 13/13 [00:03<00:00,  4.70it/s]
                                                         

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]
                                                 

Training: 100%|██████████| 13/13 [00:03<00:00,  4.59it/s]
                                                         

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  7.18it/s]
                                                         

Training: 100%|██████████| 13/13 [00:03<00:00,  4.41it/s]
                                                         

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  9.12it/s]
                                                         

Training: 100%|██████████| 13/13 [00:03<00:00,  4.77it/s]
                                                         

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  9.89it/s]
                         

## Section 5: Detect Overfitting

In [17]:
%%jshell

runPy("""
# Plot overfitting analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Overfitting Analysis: Baseline vs L2 vs Dropout', fontsize=16, fontweight='bold')

models_data = [
    ('Baseline\\n(No Regularization)', baseline_history, axes[0]),
    ('L2 Regularized\\n(Weight Decay)', l2_history, axes[1]),
    ('Dropout\\n(Dropout Rate=0.5)', dropout_history, axes[2])
]

for model_name, history, ax in models_data:
    ax.plot(history['train_acc'], label='Train Accuracy', marker='o', linewidth=2, markersize=4)
    ax.plot(history['val_acc'], label='Val Accuracy', marker='s', linewidth=2, markersize=4)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.set_title(model_name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 105])

plt.tight_layout()
plt.savefig('overfitting_comparison.png', dpi=150)
plt.savefig('/content/regression_plot.png')

print("✅ Overfitting curves plotted")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ Overfitting curves plotted


In [18]:
%%jshell

runPy("""
# Calculate overfitting metrics
overfitting_analysis = {}

baseline_gap = baseline_history['train_acc'][-1] - baseline_history['val_acc'][-1]
overfitting_analysis['Baseline'] = baseline_gap

l2_gap = l2_history['train_acc'][-1] - l2_history['val_acc'][-1]
overfitting_analysis['L2 Regularized'] = l2_gap

dropout_gap = dropout_history['train_acc'][-1] - dropout_history['val_acc'][-1]
overfitting_analysis['Dropout'] = dropout_gap

print("\\n📊 OVERFITTING ANALYSIS (Train-Val Gap):")
for model_name, gap in overfitting_analysis.items():
    print(f"{model_name:20s}: {gap:6.2f}% gap", end="")
    if gap > 10:
        print(" ⚠️  HIGH OVERFITTING")
    elif gap > 5:
        print(" ⚠️  MODERATE OVERFITTING")
    else:
        print(" ✅ GOOD GENERALIZATION")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
📊 OVERFITTING ANALYSIS (Train-Val Gap):
Baseline            :  29.62% gap ⚠️  HIGH OVERFITTING
L2 Regularized      :  38.37% gap ⚠️  HIGH OVERFITTING
Dropout             :  24.12% gap ⚠️  HIGH OVERFITTING


## Section 6: Model Complexity Analysis

In [19]:
%%jshell

runPy("""
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    torch.save(model.state_dict(), "temp_model.pth")
    size_mb = os.path.getsize("temp_model.pth") / (1024 * 1024)
    os.remove("temp_model.pth")
    return size_mb

# Calculate model sizes
model_complexity = {}

for model_name, model in [('Baseline', baseline_model), ('L2 Regularized', l2_model), ('Dropout', dropout_model)]:
    params = count_parameters(model)
    size = get_model_size_mb(model)
    model_complexity[model_name] = {'params': params, 'size': size}
    print(f"{model_name:20s}: {params:,} parameters | {size:.2f} MB")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Baseline            : 11,180,103 parameters | 42.73 MB
L2 Regularized      : 11,180,103 parameters | 42.73 MB
Dropout             : 11,180,103 parameters | 42.73 MB


## Section 7: Robustness Testing - Adversarial Attacks

In [20]:
%%jshell

runPy("""
# Gaussian Noise Perturbation
def add_gaussian_noise(images, noise_std=0.1):
    noise = torch.randn_like(images) * noise_std  # Create random noise
    noisy_images = images + noise  # Add noise to images
    return torch.clamp(noisy_images, -1, 1)  # Clip to valid range

print("✅ Gaussian noise function defined")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ Gaussian noise function defined


In [21]:
%%jshell

runPy("""
# FGSM Attack
def fgsm_attack(model, images, labels, device, epsilon=0.05):
    images.requires_grad = True  # Enable gradient tracking

    # Forward pass
    outputs = model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels)

    # Compute gradients
    model.zero_grad()
    loss.backward()

    # Get gradient sign and create adversarial examples
    data_grad = images.grad.data
    sign_data_grad = data_grad.sign()  # Sign of gradient

    # Perturb in gradient direction
    perturbed_images = images + epsilon * sign_data_grad

    # Clip to valid range
    return torch.clamp(perturbed_images, -1, 1).detach()

print("✅ FGSM attack function defined")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ FGSM attack function defined


In [22]:
%%jshell

runPy("""
# Evaluate on noisy images
def evaluate_on_noisy(model, loader, device, noise_std=0.1):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Testing noise (σ={noise_std})", leave=False):
            images, labels = images.to(device), labels.to(device)
            # Add Gaussian noise
            noisy_images = add_gaussian_noise(images, noise_std=noise_std)
            # Get predictions
            outputs = model(noisy_images)
            _, predicted = torch.max(outputs, 1)
            # Update accuracy
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

print("✅ Noisy evaluation function defined")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ Noisy evaluation function defined


In [23]:
%%jshell

runPy("""
# Evaluate on adversarial images
def evaluate_on_adversarial(model, loader, device, epsilon=0.05):
    model.eval()
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc=f"Testing FGSM (ε={epsilon})", leave=False):
        images, labels = images.to(device), labels.to(device)
        # Generate adversarial examples
        adv_images = fgsm_attack(model, images.clone(), labels, device, epsilon=epsilon)
        # Evaluate on adversarial images
        outputs = model(adv_images)
        _, predicted = torch.max(outputs, 1)
        # Update accuracy
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

print("✅ Adversarial evaluation function defined")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ Adversarial evaluation function defined


In [24]:
%%jshell

runPy("""
# Test all models on clean, noisy, and adversarial data
print("\\n" + "="*70)
print("ROBUSTNESS EVALUATION")
print("="*70)

robustness_results = {}

for model_name, model in [('Baseline', baseline_model), ('L2 Regularized', l2_model), ('Dropout', dropout_model)]:
    print(f"\\n{model_name}:")

    # Clean accuracy
    clean_loss, clean_acc = evaluate(model, test_loader, criterion, device)
    # Noisy accuracy (Gaussian noise with σ=0.1)
    noisy_acc = evaluate_on_noisy(model, test_loader, device, noise_std=0.1)
    # Adversarial accuracy (FGSM with ε=0.05)
    adv_acc = evaluate_on_adversarial(model, test_loader, device, epsilon=0.05)

    robustness_results[model_name] = {
        'clean': clean_acc,
        'noisy': noisy_acc,
        'adversarial': adv_acc
    }

    print(f"  Clean Accuracy:               {clean_acc:.2f}%")
    print(f"  Noisy Accuracy (σ=0.1):        {noisy_acc:.2f}%")
    print(f"  Adversarial Accuracy (ε=0.05): {adv_acc:.2f}%")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
ROBUSTNESS EVALUATION

Baseline:

Evaluating: 100%|██████████| 2/2 [00:00<00:00,  5.71it/s]
                                                         

Testing noise (σ=0.1): 100%|██████████| 2/2 [00:00<00:00,  5.69it/s]
                                                                    

Testing FGSM (ε=0.05): 100%|██████████| 2/2 [00:00<00:00,  3.86it/s]
                                                                    
  Clean Accuracy:               98.29%
  Noisy Accuracy (σ=0.1):        78.63%
  Adversarial Accuracy (ε=0.05): 72.65%

L2 Regularized:

Evaluating: 100%|██████████| 2/2 [00:00<00:00,  6.29it/s]
                                                         

Testing noise (σ=0.1): 100%|██████████| 2/2 [00:00<00:00,  6.10it/s]
                                                                    

Testing FGSM (ε=0.05): 100%|██████████| 2/2 [00:00<00:00,  4.31it/s]
                                         

## Section 8: Generate Final Report

In [25]:
%%jshell

runPy("""
# Create comprehensive report
print("\\n" + "="*80)
print("🎯 CNN ROBUSTNESS REPORT")
print("="*80)

print("\\n📊 ACCURACY METRICS:")
print("-" * 80)
print(f"{'Model':<25} {'Clean':<15} {'Noisy':<15} {'Adversarial':<15}")
print("-" * 80)

for model_name in ['Baseline', 'L2 Regularized', 'Dropout']:
    clean = robustness_results[model_name]['clean']
    noisy = robustness_results[model_name]['noisy']
    adv = robustness_results[model_name]['adversarial']
    print(f"{model_name:<25} {clean:>6.2f}%{'':<7} {noisy:>6.2f}%{'':<7} {adv:>6.2f}%")

print("\\n⏱️  INFERENCE TIME & MODEL SIZE:")
print("-" * 80)
print(f"{'Model':<25} {'Parameters':<20} {'Size (MB)':<15}")
print("-" * 80)

for model_name in ['Baseline', 'L2 Regularized', 'Dropout']:
    params = model_complexity[model_name]['params']
    size = model_complexity[model_name]['size']
    print(f"{model_name:<25} {params:>12,}{'':<6} {size:>6.2f}")

print("\\n🔗 GENERALIZATION & REGULARIZATION:")
print("-" * 80)
print(f"{'Model':<25} {'Train-Val Gap':<15} {'Status':<20}")
print("-" * 80)

for model_name in ['Baseline', 'L2 Regularized', 'Dropout']:
    gap = overfitting_analysis[model_name]
    if gap > 10:
        status = "High Overfitting"
    elif gap > 5:
        status = "Moderate Overfitting"
    else:
        status = "Good Generalization"
    print(f"{model_name:<25} {gap:>6.2f}%{'':<7} {status:<20}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
🎯 CNN ROBUSTNESS REPORT

📊 ACCURACY METRICS:
--------------------------------------------------------------------------------
Model                     Clean           Noisy           Adversarial    
--------------------------------------------------------------------------------
Baseline                   98.29%         78.63%         72.65%
L2 Regularized             93.16%         68.38%         58.97%
Dropout                    94.02%         80.34%         64.96%

⏱️  INFERENCE TIME & MODEL SIZE:
--------------------------------------------------------------------------------
Model                     Parameters           Size (MB)      
--------------------------------------------------------------------------------
Baseline                    11,180,103        42.73
L2 Regularized              11,180,103        42.73
Dropout                     11,180,103        42.73

🔗 GENERALIZATION & REGULARIZATION:
-------

In [26]:
%%jshell

runPy("""
# Create visualization
fig, ax = plt.subplots(figsize=(12, 6))

model_names = ['Baseline', 'L2 Regularized', 'Dropout']
x = np.arange(len(model_names))
width = 0.25

clean_accs = [robustness_results[m]['clean'] for m in model_names]
noisy_accs = [robustness_results[m]['noisy'] for m in model_names]
adv_accs = [robustness_results[m]['adversarial'] for m in model_names]

ax.bar(x - width, clean_accs, width, label='Clean', alpha=0.8, color='#2ecc71')
ax.bar(x, noisy_accs, width, label='Noisy (σ=0.1)', alpha=0.8, color='#f39c12')
ax.bar(x + width, adv_accs, width, label='Adversarial (ε=0.05)', alpha=0.8, color='#e74c3c')

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Robustness Comparison: Clean vs Noisy vs Adversarial', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 105])

# Add value labels on bars
for bars in [ax.patches[i::len(model_names)] for i in range(len(model_names))]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('robustness_summary.png', dpi=150, bbox_inches='tight')
plt.savefig('/content/regression_plot.png')

print("✅ Summary visualization created")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ✅ Summary visualization created


## Key Findings & Analysis

In [27]:
%%jshell

runPy("""
print("\\n" + "="*80)
print("💡 OBSERVATIONS & ANALYSIS")
print("="*80)

print("\\n1. OVERFITTING ANALYSIS:")
print("   - The Baseline model shows the highest train-val gap (most overfitting)")
print(f"     Baseline gap: {overfitting_analysis['Baseline']:.2f}%")
print(f"     L2 Regularized gap: {overfitting_analysis['L2 Regularized']:.2f}%")
print(f"     Dropout gap: {overfitting_analysis['Dropout']:.2f}%")
print("   This is expected because we trained without any regularization.")

print("\\n2. REGULARIZATION EFFECTIVENESS:")
l2_effect = overfitting_analysis['Baseline'] - overfitting_analysis['L2 Regularized']
dropout_effect = overfitting_analysis['Baseline'] - overfitting_analysis['Dropout']
print(f"   - L2 Regularization reduced overfitting gap by {l2_effect:.2f}%")
print(f"   - Dropout reduced overfitting gap by {dropout_effect:.2f}%")
if dropout_effect > l2_effect:
    print("   → Dropout is more effective at controlling overfitting")
else:
    print("   → L2 Regularization is more effective at controlling overfitting")

print("\\n3. ROBUSTNESS ASSESSMENT:")
for model_name in model_names:
    clean = robustness_results[model_name]['clean']
    adv = robustness_results[model_name]['adversarial']
    drop = clean - adv
    print(f"   - {model_name}: Clean {clean:.2f}% → Adversarial {adv:.2f}% (drop: {drop:.2f}%)")

print("\\n4. ACCURACY-COMPLEXITY TRADEOFF:")
for model_name in model_names:
    acc = robustness_results[model_name]['clean']
    size = model_complexity[model_name]['size']
    print(f"   - {model_name}: {acc:.2f}% accuracy, {size:.2f} MB model size")
print("   → Model size is identical across all three (same architecture)")
print("   → Performance differs due to regularization, not model complexity")

print("\\n5. RECOMMENDATIONS:")
best_model = max(model_names, key=lambda m: robustness_results[m]['clean'])
best_robust = min(model_names, key=lambda m: robustness_results[m]['clean'] - robustness_results[m]['adversarial'])
print(f"   - For highest clean accuracy: {best_model}")
print(f"   - For best adversarial robustness: {best_robust}")
print(f"   - Deploy {best_robust} in production for better robustness")
print("\\n" + "="*80)
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
💡 OBSERVATIONS & ANALYSIS

1. OVERFITTING ANALYSIS:
   - The Baseline model shows the highest train-val gap (most overfitting)
     Baseline gap: 29.62%
     L2 Regularized gap: 38.37%
     Dropout gap: 24.12%
   This is expected because we trained without any regularization.

2. REGULARIZATION EFFECTIVENESS:
   - L2 Regularization reduced overfitting gap by -8.74%
   - Dropout reduced overfitting gap by 5.50%
   → Dropout is more effective at controlling overfitting

3. ROBUSTNESS ASSESSMENT:
   - Baseline: Clean 98.29% → Adversarial 72.65% (drop: 25.64%)
   - L2 Regularized: Clean 93.16% → Adversarial 58.97% (drop: 34.19%)
   - Dropout: Clean 94.02% → Adversarial 64.96% (drop: 29.06%)

4. ACCURACY-COMPLEXITY TRADEOFF:
   - Baseline: 98.29% accuracy, 42.73 MB model size
   - L2 Regularized: 93.16% accuracy, 42.73 MB model size
   - Dropout: 94.02% accuracy, 42.73 MB model size
   → Model size is identical across all 

## Summary Table

In [28]:
%%jshell

runPy("""
import pandas as pd

# Create comprehensive summary table
summary_data = {
    'Model': model_names,
    'Clean Acc (%)': [robustness_results[m]['clean'] for m in model_names],
    'Noisy Acc (%)': [robustness_results[m]['noisy'] for m in model_names],
    'Adversarial Acc (%)': [robustness_results[m]['adversarial'] for m in model_names],
    'Overfitting Gap (%)': [overfitting_analysis[m] for m in model_names],
    'Model Size (MB)': [model_complexity[m]['size'] for m in model_names],
    'Parameters': [model_complexity[m]['params'] for m in model_names]
}

df = pd.DataFrame(summary_data)
print("\\n📋 COMPREHENSIVE SUMMARY TABLE:")
print(df.to_string(index=False))
print("\\n✅ Lab 1 Complete!")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
📋 COMPREHENSIVE SUMMARY TABLE:
         Model  Clean Acc (%)  Noisy Acc (%)  Adversarial Acc (%)  Overfitting Gap (%)  Model Size (MB)  Parameters
      Baseline      98.290598      78.632479            72.649573            29.623116        42.725902    11180103
L2 Regularized      93.162393      68.376068            58.974359            38.366834        42.725902    11180103
       Dropout      94.017094      80.341880            64.957265            24.120603        42.725963    11180103

✅ Lab 1 Complete!


In [29]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-29 16:34:42
